In [1]:
%pwd

'/home/phil/Coding/thesis-code/arco/notebooks'

In [2]:
%cd ..

/home/phil/Coding/thesis-code/arco


In [3]:
# Load OPENAI API KEY from the keychain
import os
import subprocess

os.environ["OPENAI_API_KEY"] = subprocess.check_output(
    ["secret-tool", "lookup", "app", "thesis", "provider", "openai"], text=True
)
os.environ["OPENROUTER_API_KEY"] = subprocess.check_output(
    ["secret-tool", "lookup", "app", "thesis", "provider", "openrouter"], text=True
)

In [4]:
from typing import override

from arco.cli.viz import display_workflow_notebook
from arco.core import LLM, Agent, Config, Graph, State, Workflow
from arco.core.graph import END

In [5]:
class CodingAgent(Agent):
    CODING_PROMPT = """You're a coding agent.\nGiven a question, firstly reply with a short reasoning on what you would like to implement, then report the implementation and finally comment on specific details on that implementation\n\n##QUESTION\n{prompt}"""

    @override
    def core(self, state: State, llm: LLM) -> State:
        output = llm.invoke(CodingAgent.CODING_PROMPT.format(prompt=state.prompt))
        return self.answer(
            state,
            message="Code has been generated",
            output={"code": output.text},
            logprobs=output.logprobs,
        )


class ScoringAgent(Agent):
    SCORING_PROMPT = """You're a scoring agent.\nGiven code, firstly reply with a short reasoning on how you would like to score the input as a series of well-defined criteria, then report the scoring values for each criteria. Format the scoring as a JSON\n\n##INPUT\n{last_output}"""

    @override
    def core(self, state: State, llm: LLM) -> State:
        last_output: str = state.get_last_answer().agent_output
        output = llm.invoke(ScoringAgent.SCORING_PROMPT.format(last_output=last_output))
        return self.answer(
            state,
            message=f"The evaluation is : {output.text}",
            logprobs=output.logprobs,
        )


class MyWorkflow(Workflow):
    workflow_id: str = "my_workflow"

    @override
    def initialize(self, config: Config, graph: Graph):
        coding_agent = CodingAgent()
        scoring_agent = ScoringAgent()

        graph.add_agent(coding_agent)
        graph.add_agent(scoring_agent)

        graph.set_entry_agent(coding_agent)
        graph.add_agent_edge(coding_agent, scoring_agent)
        graph.add_agent_edge(scoring_agent, END)


workflow = MyWorkflow()
print(workflow)

  +-----------+  
  | __start__ |  
  +-----------+  
        *        
        *        
        *        
+-------------+  
| CodingAgent |  
+-------------+  
        *        
        *        
        *        
+--------------+ 
| ScoringAgent | 
+--------------+ 
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    


In [6]:
import difflib

from arco.core import Evaluation, Evaluator


class CodingEvaluator(Evaluator):
    def _eval(self, state: State, judge_provider: str, judge_model: str):
        pass

    def _batch_eval(self, states: list[State]) -> bool:
        return False

    def _gt_eval(self, answer, gt_data, judge_provider, judge_model):
        generated = answer.agent_output.get("code", "")
        expected = (gt_data or {}).get("expected_code", "")
        score = difflib.SequenceMatcher(None, generated, expected).ratio()
        answer.gt_evaluation = Evaluation(score=score, success=score > 0.5)


class ScoringEvaluator(Evaluator):
    def _eval(self, state: State, judge_provider: str, judge_model: str):
        pass

    def _batch_eval(self, states: list[State]) -> bool:
        return False

    def _gt_eval(self, answer, gt_data, judge_provider, judge_model):
        if answer.message and len(answer.message) > 10:
            answer.gt_evaluation = Evaluation(score=1.0, success=True)
        else:
            answer.gt_evaluation = Evaluation(score=0.0, success=False)


# Wire evaluators into the existing agent classes
CodingAgent.evaluator = CodingEvaluator()
ScoringAgent.evaluator = ScoringEvaluator()

In [7]:
final_state: State = display_workflow_notebook(workflow.stream(), verbose=True)

🚀 Run `weighted-buffer-741`

⏳ Checking 1 model(s)…

▶ **CodingAgent**

✅ **CodingAgent**

▶ **ScoringAgent**

✅ **ScoringAgent**

✅ Completed — total time 10.62s

In [9]:
from arco.tools import benchmark_from_config

CONFIG = "./config/benchmark_config/my_workflow_example.yaml"
DATASET = "./data/benchmarks/my_workflow_example.json"
SAVE_DIR = "./output/benchmarks"


def _collect_state(events):
    """Minimal visualization callback: runs the stream and returns final state."""
    state = None
    for event in events:
        event_type = event["event"]
        if event_type == "started":
            print(f"  Run {event.get('run_id', '?')}")
        elif event_type == "node_started":
            print(f"    \u25b6 {event['node']}")
        elif event_type == "node_finished":
            print(f"    \u2705 {event['node']}")
        elif event_type == "completed":
            state = event["state"]
            print(f"    \u2705 Done — {state.global_profiling_data.total_time:.2f}s")
        elif event_type == "error":
            print(f"    \u274c {event.get('message', '?')}")
    return state


for event in benchmark_from_config(
    config_path=CONFIG,
    dataset_path=DATASET,
    id=None,
    save_dir=SAVE_DIR,
    logging_level=None,
    run_visualization_logic=_collect_state,
):
    event_type = event["event"]
    if event_type == "run_configs_loaded":
        print(f"Loaded {len(event['configs'])} run config(s)")
    elif event_type == "benchmark_start":
        print(f"\n=== {event['name']}: {event['description']} ===")
    elif event_type == "test_case_start":
        print(f"\n[{event['iteration']}/{event['max_iteration']}]")
    elif event_type == "benchmark_run_save":
        print(f"Results saved to {event['path']}")
    elif event_type == "aggregated_summary_save":
        print(f"\nSummary saved to {event['path']}")

Loaded 1 run config(s)

=== Baseline: Single run with default parameters ===

[1/2]
  Run plotting-nexus-514
    ▶ CodingAgent
    ✅ CodingAgent
    ▶ ScoringAgent
    ✅ ScoringAgent
    ✅ Done — 10.78s

[2/2]
  Run augmented-chart-812
    ▶ CodingAgent
    ✅ CodingAgent
    ▶ ScoringAgent
    ✅ ScoringAgent
    ✅ Done — 12.10s
Results saved to output/benchmarks/my_workflow_example/runs/Baseline/Baseline.csv

Summary saved to output/benchmarks/my_workflow_example/summary.csv
